# Stream clickstream events from a producer script to S3 via Kinesis Data Firehose

Marketing emailed at 4 PM. They want a clickstream feed in S3 by tomorrow morning so the data science team can run partitioned Athena queries on the JSON events. You have until end of day to build the Firehose pipeline, prove events land in S3 within a minute, and confirm the partition layout matches what data science asked for.

The producer is already written and will emit 500 JSON events tagged with one of three `event_type` values: `page_view`, `add_to_cart`, `purchase`. Your job is the pipe in the middle. Build an IAM role Firehose can assume, create a delivery stream with dynamic partitioning by `event_type` and UTC date, enable Snappy compression so the cold-storage bill stays small, and verify that every event lands at the right partition prefix.

The four deliverables map to four checkpoints:
1. A Firehose service role with the Firehose trust policy and an inline policy scoped to one bucket and one log group.
2. A delivery stream with dynamic partitioning, JSON metadata extraction, and Snappy compression, pointing at the lab bucket.
3. 500 events produced and confirmed in S3 inside the 60-second buffer hint.
4. Partition layout matches `event_type=<value>/dt=YYYY-MM-DD/` and every event_type bucket has at least one object.

**Time.** Plan for about 55 minutes hands-on. Most of the wall-clock time is the 60-second Firehose buffer flush, not your typing. If you fight an IAM trust-policy bug, budget up to 90 minutes total.

**Cost.** This lab costs about a penny if everything works first try. Firehose bills by the GB ingested and you are streaming kilobytes. Add a nickel if you re-run the producer a few times after fixing the role. The cleanup cell at the end deletes the stream so even the tiny bill stops the moment you finish.

This lab maps to AWS DEA-C01 Domain 1: Data Ingestion and Transformation (34% of exam weight).


In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.7.0


In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12.

import atexit
import getpass
import json
import random
import re
import time
from datetime import datetime, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "kinesis-streaming-ingestion"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[2].slug exactly
REGION = "us-east-1"  # all DEA-C01 labs run in us-east-1 per LAB-CREATION-BLUEPRINT section 15

# Resource names. Bucket name appends the account ID for global uniqueness.
ROLE_NAME = f"labrun-{LAB_ID}-firehose-role"
LOG_GROUP_NAME = f"/aws/kinesisfirehose/labrun-{LAB_ID}"
STREAM_NAME = f"labrun-{LAB_ID}-stream"
INLINE_POLICY_NAME = f"labrun-{LAB_ID}-firehose-inline"
BUCKET_NAME = None  # resolved after STS once the account ID is known

EVENT_TYPES = ["page_view", "add_to_cart", "purchase"]

# Producer-start timestamp is set when the producer cell runs. Checkpoint 3
# reads it to compute end-to-end latency.
PRODUCER_START_TIME = None


In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"All DEA-C01 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

# Validate credentials against AWS via STS GetCallerIdentity. Fail fast with a
# clear message rather than waiting for the first IAM call to error out.
sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

# Resolve the bucket name now that we know the account ID. S3 bucket names
# must be globally unique.
BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

# Register the session with labrun-checks. register() returns None;
# do not assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# The manifest is module-level and in reverse-creation order. Lab 03 has no
# critical (hourly-billed) resources. Firehose bills per GB ingested and the
# lab emits well under 1 MB, so every entry is standard priority.
#
# Per RESOURCE-SAFETY-SPEC Rule 4, the orphan scan blocks execution if any
# tagged resources from a prior session exist (not just print a warning).
#
# Note: labrun-checks 0.3.0 ships AWS adapters for s3_bucket and iam_role.
# It does NOT yet support firehose_delivery_stream or cloudwatch_log_group.
# A LabAdapter subclass extending AwsCleanupAdapter is defined in the cleanup
# cell at the bottom of this notebook and passed to run_cleanup. The two new
# resource types in the manifest below are still declared declaratively here
# so the canonical summary, atexit handler, and orphan scan all see them.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="firehose_delivery_stream",
        id=STREAM_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws firehose delete-delivery-stream "
            f"--delivery-stream-name {STREAM_NAME} --allow-force-delete"
        ),
    ),
    CleanupResource(
        type="iam_role",
        id=ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {ROLE_NAME}",
    ),
    CleanupResource(
        type="cloudwatch_log_group",
        id=LOG_GROUP_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws logs delete-log-group --log-group-name {LOG_GROUP_NAME}"
        ),
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    The dedicated cleanup cell is the authoritative entry point; this is the
    safety net for kernel crashes. Errors are swallowed because atexit
    handlers must not raise.
    """
    try:
        # Late import to avoid circulars at notebook load time.
        from labrun_checks.adapters.aws import AwsCleanupAdapter
        run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter()) if "_LabAdapter" in globals() else run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell must check for leftover
    state with the lab's tag and surface the cleanup command before creating
    any new resources.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources or")
    print("collide on the role and stream names. Run the cleanup cell at the")
    print("bottom of this notebook first, or remove them manually with the")
    print("AWS CLI commands the cleanup cell prints, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Task 1: Create the destination bucket and the Firehose IAM role

Firehose is a managed service that needs to assume an IAM role in your account before it can write to your S3 bucket and your CloudWatch log group. You build the role; AWS Firehose assumes it on every delivery.

Two pieces to assemble in this task:
1. The S3 bucket Firehose will write into. Name `labrun-kinesis-streaming-ingestion-{your-account-id}`, tagged with `labrun:lab-slug=kinesis-streaming-ingestion`.
2. The CloudWatch log group for Firehose error logs at `/aws/kinesisfirehose/labrun-kinesis-streaming-ingestion`. Tag it with the same lab slug so the orphan scan can find it later.
3. The IAM role `labrun-kinesis-streaming-ingestion-firehose-role` with:
   - Trust policy: `Principal.Service` is `firehose.amazonaws.com`, `Action` is `sts:AssumeRole`.
   - One inline policy named `labrun-kinesis-streaming-ingestion-firehose-inline` that grants the six S3 actions (`AbortMultipartUpload`, `GetBucketLocation`, `GetObject`, `ListBucket`, `ListBucketMultipartUploads`, `PutObject`) on the lab bucket and its `/*` ARN, plus `logs:PutLogEvents` on the log group's log-stream ARN.

The Firehose trust policy is what makes this an IAM role and not just any policy bundle. The service that assumes the role is Firehose itself, not a user, not an EC2 instance. Get the Principal wrong and Firehose returns `InvalidArgumentException` at delivery-stream-create time.

After creating the role, sleep 15 seconds before the next cell. IAM role propagation is asynchronous and Firehose validates the role at delivery-stream-create. Skipping the sleep is the most common false-FAIL on Checkpoint 2.


In [ ]:
# NBVAL_SKIP
# Task 1: Create the bucket, the Firehose log group, and the Firehose IAM
# role with its inline policy.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
logs = boto3.client(
    "logs",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Create the bucket. us-east-1 rejects LocationConstraint; other regions
# require it. All DEA-C01 labs run in us-east-1, so the simple call works.
# YOUR CODE: Create the S3 bucket with s3.create_bucket(Bucket=BUCKET_NAME).

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

# Create the Firehose error log group and tag it with the lab slug so the
# orphan scan and Phase 3 tag audit can find it later.
# YOUR CODE: Create the log group with logs.create_log_group(logGroupName=LOG_GROUP_NAME).
try:
    logs.tag_resource(
        resourceArn=f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{LOG_GROUP_NAME}",
        tags={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
except ClientError as e:
    # If the log group already exists from a prior run, the orphan scan would
    # have caught it. If we are here, the tagging API surface is fine to retry.
    if e.response["Error"]["Code"] != "ResourceNotFoundException":
        raise
print(f"Log group created and tagged: {LOG_GROUP_NAME}")

# Build the Firehose trust policy. The Principal is the Firehose service.
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "firehose.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

# YOUR CODE: Create the IAM role with iam.create_role(
#   RoleName=ROLE_NAME,
#   AssumeRolePolicyDocument=json.dumps(trust_policy),
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).

# Build the inline policy. Six S3 actions on the bucket + /* ARNs, plus
# logs:PutLogEvents scoped to this lab's log group log-stream ARN.
bucket_arn = f"arn:aws:s3:::{BUCKET_NAME}"
log_stream_arn = (
    f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{LOG_GROUP_NAME}:log-stream:*"
)
inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:AbortMultipartUpload",
                "s3:GetBucketLocation",
                "s3:GetObject",
                "s3:ListBucket",
                "s3:ListBucketMultipartUploads",
                "s3:PutObject",
            ],
            "Resource": [bucket_arn, f"{bucket_arn}/*"],
        },
        {
            "Effect": "Allow",
            "Action": ["logs:PutLogEvents"],
            "Resource": log_stream_arn,
        },
    ],
}

# YOUR CODE: Attach the inline policy with iam.put_role_policy(
#   RoleName=ROLE_NAME,
#   PolicyName=INLINE_POLICY_NAME,
#   PolicyDocument=json.dumps(inline_policy),
# ).

print(f"Role created: {ROLE_NAME}")
print("Inline policy attached.")

# IAM role propagation. Firehose validates the role at create-stream time.
# Skipping this sleep is the most common false-FAIL on Checkpoint 2.
print("Your IAM role is stuck in traffic, give it 15 seconds...")
time.sleep(15)
print("Role is ready.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Firehose IAM role exists with the correct trust policy and
# inline policy. The validator accepts s3:* / * wildcards alongside the
# six literal required actions and checks the full required-action set.

REQUIRED_S3_ACTIONS = {
    "s3:AbortMultipartUpload",
    "s3:GetBucketLocation",
    "s3:GetObject",
    "s3:ListBucket",
    "s3:ListBucketMultipartUploads",
    "s3:PutObject",
}


def _action_covers(action: str, required: str) -> bool:
    """True if an action string in a policy covers the required action.

    Accepts literal match, '*', 's3:*'-style service wildcard, and any
    service-scoped wildcard whose prefix matches the required action's
    service segment.
    """
    if action == required:
        return True
    if action == "*":
        return True
    if ":" not in required:
        return False
    service = required.split(":", 1)[0]
    if action == f"{service}:*":
        return True
    return False


def _collect_actions(policy_doc: dict) -> set[str]:
    actions: set[str] = set()
    statements = policy_doc.get("Statement", [])
    if isinstance(statements, dict):
        statements = [statements]
    for stmt in statements:
        if stmt.get("Effect") != "Allow":
            continue
        raw = stmt.get("Action", [])
        if isinstance(raw, str):
            actions.add(raw)
        else:
            actions.update(raw)
    return actions


def checkpoint_1(session):
    try:
        iam_client = boto3.client(
            "iam",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        # Role exists and trust policy names firehose.amazonaws.com.
        try:
            role = iam_client.get_role(RoleName=ROLE_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "NoSuchEntity":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Role {ROLE_NAME} does not exist.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        trust = role["Role"]["AssumeRolePolicyDocument"]
        if isinstance(trust, str):
            trust = json.loads(trust)
        principal_services: set[str] = set()
        for stmt in trust.get("Statement", []):
            principal = stmt.get("Principal", {})
            svc = principal.get("Service")
            if isinstance(svc, str):
                principal_services.add(svc)
            elif isinstance(svc, list):
                principal_services.update(svc)
        if "firehose.amazonaws.com" not in principal_services:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Trust policy on {ROLE_NAME} does not name "
                    f"firehose.amazonaws.com as a service Principal. "
                    f"Found services: {sorted(principal_services) or 'none'}."
                ),
            )

        # Inline policy exists and grants the required action set on the
        # right resources.
        try:
            inline = iam_client.list_role_policies(RoleName=ROLE_NAME)
        except ClientError as e:
            return CheckpointResult(status="error", error_reason=str(e))
        if not inline.get("PolicyNames"):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Role {ROLE_NAME} has no inline policy. Attach one with "
                    f"iam.put_role_policy."
                ),
            )

        all_actions: set[str] = set()
        resources_seen: set[str] = set()
        logs_actions: set[str] = set()
        logs_resources_seen: set[str] = set()
        for policy_name in inline["PolicyNames"]:
            doc = iam_client.get_role_policy(
                RoleName=ROLE_NAME, PolicyName=policy_name
            )["PolicyDocument"]
            if isinstance(doc, str):
                doc = json.loads(doc)
            for stmt in doc.get("Statement", []):
                if stmt.get("Effect") != "Allow":
                    continue
                raw_actions = stmt.get("Action", [])
                if isinstance(raw_actions, str):
                    raw_actions = [raw_actions]
                raw_resources = stmt.get("Resource", [])
                if isinstance(raw_resources, str):
                    raw_resources = [raw_resources]
                # Statements that mention any logs:* action go into the logs bucket.
                if any(a == "*" or a.startswith("logs:") for a in raw_actions):
                    logs_actions.update(raw_actions)
                    logs_resources_seen.update(raw_resources)
                if any(a == "*" or a.startswith("s3:") for a in raw_actions):
                    all_actions.update(raw_actions)
                    resources_seen.update(raw_resources)

        # S3 action coverage: every required action must be literally present
        # or covered by a wildcard.
        missing = [
            req for req in REQUIRED_S3_ACTIONS
            if not any(_action_covers(a, req) for a in all_actions)
        ]
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Inline policy is missing S3 actions: {sorted(missing)}. "
                    f"Found: {sorted(all_actions)}. A wildcard like s3:* also "
                    f"satisfies the requirement."
                ),
            )

        # S3 resource coverage: both bucket ARN and bucket/* ARN must appear.
        bucket_arn = f"arn:aws:s3:::{BUCKET_NAME}"
        wanted_resources = {bucket_arn, f"{bucket_arn}/*"}
        if not wanted_resources.issubset(resources_seen):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Inline policy must list both {bucket_arn!r} and "
                    f"{bucket_arn + '/*'!r} as resources. Found: "
                    f"{sorted(resources_seen)}."
                ),
            )

        # Logs action coverage: at minimum logs:PutLogEvents or a wildcard.
        if not any(_action_covers(a, "logs:PutLogEvents") for a in logs_actions):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Inline policy is missing logs:PutLogEvents (or a wildcard "
                    "covering it). Firehose needs this to write its error logs."
                ),
            )

        # Logs resource coverage: at minimum a log-group or log-stream ARN that
        # mentions this lab's log group name.
        if not any(LOG_GROUP_NAME in r or r == "*" for r in logs_resources_seen):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Inline policy must reference the log group "
                    f"{LOG_GROUP_NAME} in a logs:PutLogEvents resource ARN. "
                    f"Found resources: {sorted(logs_resources_seen)}."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint says the role exists but something about the policies is off. Read the failure reason carefully. The two layers it checks are the trust policy (who can assume) and the inline policy (what the assumed role can do). The Firehose service-role pattern requires both, and the inline policy needs two distinct statements because S3 and CloudWatch Logs are different services.

</details>

<details><summary>Hint 2 (direction)</summary>

The trust policy must name `firehose.amazonaws.com` as a `Principal.Service`. Not `firehose`, not `amazonaws.com`, not a wildcard. Exact string. The inline policy needs one statement granting the six S3 actions on both the bucket ARN and the `bucket/*` ARN, plus a separate statement granting `logs:PutLogEvents` on the log-stream ARN under this lab's log group. A wildcard `s3:*` satisfies the S3 actions check, but the resource ARNs still must match.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1:

```python
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
logs = boto3.client(
    "logs",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

s3.create_bucket(Bucket=BUCKET_NAME)
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)

logs.create_log_group(logGroupName=LOG_GROUP_NAME)
logs.tag_resource(
    resourceArn=f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{LOG_GROUP_NAME}",
    tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "firehose.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}
iam.create_role(
    RoleName=ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

bucket_arn = f"arn:aws:s3:::{BUCKET_NAME}"
log_stream_arn = (
    f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{LOG_GROUP_NAME}:log-stream:*"
)
inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:AbortMultipartUpload",
                "s3:GetBucketLocation",
                "s3:GetObject",
                "s3:ListBucket",
                "s3:ListBucketMultipartUploads",
                "s3:PutObject",
            ],
            "Resource": [bucket_arn, f"{bucket_arn}/*"],
        },
        {
            "Effect": "Allow",
            "Action": ["logs:PutLogEvents"],
            "Resource": log_stream_arn,
        },
    ],
}
iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName=INLINE_POLICY_NAME,
    PolicyDocument=json.dumps(inline_policy),
)

time.sleep(15)
```

The 15-second sleep is the IAM-to-Firehose propagation wait. Skipping it produces `InvalidArgumentException: The role ARN ... is not valid` at create-delivery-stream time even though the role exists. The bug always looks like a stream problem; the cause is always propagation.

</details>


**Wallet check.** Still at $0.00. Bucket creation, an empty log group, an IAM role, and one inline policy fit inside the always-free S3 and IAM tiers. CloudWatch Logs ingestion-and-storage budget is 5 GB per month free and Firehose has not written anything yet. Your morning coffee cost the entire AWS bill of this checkpoint, twice over.


## Task 2: Create the Firehose delivery stream with dynamic partitioning and Snappy compression

Now wire the role into a Firehose delivery stream that writes JSON events to S3 under `event_type=<value>/dt=YYYY-MM-DD/`. Three things have to be set together or the partitioning silently misroutes:

1. `DeliveryStreamType="DirectPut"`. The producer writes directly via `put_record_batch`. No Kinesis Data Stream source.
2. `DynamicPartitioningConfiguration={"Enabled": True}`. Without this the prefix tokens never resolve.
3. A `ProcessingConfiguration` with one `MetadataExtraction` processor that reads `event_type` from each JSON record via JQ. The two parameters are `MetadataExtractionQuery` (a JQ expression like `{event_type: .event_type}`) and `JsonParsingEngine: "JQ-1.6"`.

The S3 destination `Prefix` carries the dynamic-partition expression literally:

```
event_type=!{partitionKeyFromQuery:event_type}/dt=!{timestamp:yyyy-MM-dd}/
```

The `!{partitionKeyFromQuery:event_type}` token resolves to whatever the JQ query extracted for that record. The `!{timestamp:yyyy-MM-dd}` token resolves to the UTC date when Firehose flushes the buffer. Without the JQ processor the first token never resolves, every event lands under the literal string `!{partitionKeyFromQuery:event_type}`, and Checkpoint 4 catches it as a regex miss.

Set `CompressionFormat="Snappy"` so the cold-storage bill stays small. Set `BufferingHints.IntervalInSeconds=60`. Firehose with dynamic partitioning rejects buffer intervals under 60.

Also pass `Tags=[{"Key": "labrun:lab-slug", "Value": "kinesis-streaming-ingestion"}]` so the orphan scan and Phase 3 tag audit can find the stream later.

The create call returns fast but the stream is not actually `ACTIVE` for 30 to 90 seconds. Poll `firehose.describe_delivery_stream` until `DeliveryStreamStatus == "ACTIVE"` before running the producer.


In [ ]:
# NBVAL_SKIP
# Task 2: Create the Firehose delivery stream with dynamic partitioning,
# JQ metadata extraction, and Snappy compression.

firehose = boto3.client(
    "firehose",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

bucket_arn = f"arn:aws:s3:::{BUCKET_NAME}"
role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"

prefix = "event_type=!{partitionKeyFromQuery:event_type}/dt=!{timestamp:yyyy-MM-dd}/"
error_output_prefix = "errors/!{firehose:error-output-type}/dt=!{timestamp:yyyy-MM-dd}/"

extended_s3_config = {
    "RoleARN": role_arn,
    "BucketARN": bucket_arn,
    "Prefix": prefix,
    "ErrorOutputPrefix": error_output_prefix,
    "CompressionFormat": "Snappy",
    "BufferingHints": {"SizeInMBs": 64, "IntervalInSeconds": 60},
    "DynamicPartitioningConfiguration": {
        "Enabled": True,
        "RetryOptions": {"DurationInSeconds": 300},
    },
    "ProcessingConfiguration": {
        "Enabled": True,
        "Processors": [
            {
                "Type": "MetadataExtraction",
                "Parameters": [
                    {
                        "ParameterName": "MetadataExtractionQuery",
                        "ParameterValue": "{event_type: .event_type}",
                    },
                    {"ParameterName": "JsonParsingEngine", "ParameterValue": "JQ-1.6"},
                ],
            }
        ],
    },
    "CloudWatchLoggingOptions": {
        "Enabled": True,
        "LogGroupName": LOG_GROUP_NAME,
        "LogStreamName": "S3Delivery",
    },
}

# YOUR CODE: Create the delivery stream with firehose.create_delivery_stream(
#   DeliveryStreamName=STREAM_NAME,
#   DeliveryStreamType="DirectPut",
#   ExtendedS3DestinationConfiguration=extended_s3_config,
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).

print(f"Stream submitted: {STREAM_NAME}")
print("Asking Firehose to put on her party hat...")

# Poll until ACTIVE. Firehose takes 30 to 90 seconds.
deadline = time.time() + 180
while time.time() < deadline:
    desc = firehose.describe_delivery_stream(DeliveryStreamName=STREAM_NAME)
    status = desc["DeliveryStreamDescription"]["DeliveryStreamStatus"]
    if status == "ACTIVE":
        break
    print(f"  status: {status}, waiting...")
    time.sleep(10)
else:
    raise RuntimeError(
        f"Stream {STREAM_NAME} did not reach ACTIVE within 180 seconds. "
        f"Check the AWS console for a CREATING_FAILED status and inspect "
        f"the role trust policy."
    )

print(f"Stream is ACTIVE: {STREAM_NAME}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Firehose delivery stream is ACTIVE with the correct
# destination, dynamic partitioning enabled, Snappy compression, the JQ
# metadata-extraction processor, and a Prefix that references both
# event_type and a yyyy-MM-dd date token.

def checkpoint_2(session):
    try:
        firehose_client = boto3.client(
            "firehose",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            desc = firehose_client.describe_delivery_stream(
                DeliveryStreamName=STREAM_NAME
            )
        except ClientError as e:
            code = e.response["Error"]["Code"]
            if code in ("ResourceNotFoundException", "NoSuchEntity"):
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Delivery stream {STREAM_NAME} does not exist.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        stream_desc = desc["DeliveryStreamDescription"]
        status_name = stream_desc["DeliveryStreamStatus"]
        if status_name != "ACTIVE":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Stream {STREAM_NAME} is {status_name!r}. Wait until "
                    f"DeliveryStreamStatus is ACTIVE before re-running."
                ),
            )

        destinations = stream_desc.get("Destinations", [])
        if not destinations:
            return CheckpointResult(
                status="fail",
                error_reason=f"Stream {STREAM_NAME} has no destinations.",
            )
        dest = destinations[0]

        # ExtendedS3DestinationDescription should be present and point at the
        # lab bucket.
        ext = dest.get("ExtendedS3DestinationDescription")
        if not ext:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Destination is not an ExtendedS3DestinationDescription. "
                    "Create the stream with ExtendedS3DestinationConfiguration, "
                    "not S3DestinationConfiguration."
                ),
            )
        bucket_arn_seen = ext.get("BucketARN", "")
        wanted_bucket = f"arn:aws:s3:::{BUCKET_NAME}"
        if bucket_arn_seen != wanted_bucket:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Destination BucketARN is {bucket_arn_seen!r}, "
                    f"expected {wanted_bucket!r}."
                ),
            )

        # Dynamic partitioning enabled.
        dyn = ext.get("DynamicPartitioningConfiguration", {})
        if not dyn.get("Enabled"):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "DynamicPartitioningConfiguration.Enabled is not True. "
                    "Set it to True at create-stream time; it cannot be flipped "
                    "after the fact on an existing stream."
                ),
            )

        # Snappy compression. The API normalizes the response casing.
        comp = ext.get("CompressionFormat", "")
        if comp.lower() != "snappy":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"CompressionFormat is {comp!r}, expected Snappy."
                ),
            )

        # Prefix references event_type and a yyyy-MM-dd date token.
        prefix_seen = ext.get("Prefix", "")
        if "event_type" not in prefix_seen:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Prefix {prefix_seen!r} does not reference event_type. "
                    f"Use event_type=!{{partitionKeyFromQuery:event_type}}/ at "
                    f"the start of the Prefix."
                ),
            )
        if "yyyy-MM-dd" not in prefix_seen:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Prefix {prefix_seen!r} does not reference a yyyy-MM-dd "
                    f"date token. Add /dt=!{{timestamp:yyyy-MM-dd}}/ after the "
                    f"event_type segment."
                ),
            )

        # MetadataExtraction processor with JQ-1.6.
        proc_config = ext.get("ProcessingConfiguration", {})
        if not proc_config.get("Enabled"):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "ProcessingConfiguration.Enabled is not True. Dynamic "
                    "partitioning by event_type needs a MetadataExtraction "
                    "processor with the JsonParsingEngine parameter set to "
                    "JQ-1.6."
                ),
            )
        processors = proc_config.get("Processors", [])
        has_metadata_extraction = False
        has_jq = False
        for proc in processors:
            if proc.get("Type") != "MetadataExtraction":
                continue
            has_metadata_extraction = True
            for param in proc.get("Parameters", []):
                name = param.get("ParameterName")
                value = param.get("ParameterValue", "")
                if name == "JsonParsingEngine" and value.upper().startswith("JQ"):
                    has_jq = True
        if not has_metadata_extraction:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "No MetadataExtraction processor found. Dynamic "
                    "partitioning on event_type needs one."
                ),
            )
        if not has_jq:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "MetadataExtraction processor is missing the "
                    "JsonParsingEngine=JQ-1.6 parameter."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks the delivery stream description in order: status, destination, dynamic partitioning, Snappy, Prefix tokens, MetadataExtraction processor. Read the failure reason. The most common miss is omitting the JQ processor; without it the prefix token cannot resolve. The next most common miss is forgetting to flip `DynamicPartitioningConfiguration.Enabled` to True, which silently leaves the prefix tokens as literal strings.

</details>

<details><summary>Hint 2 (direction)</summary>

The Firehose create call needs an `ExtendedS3DestinationConfiguration` (not `S3DestinationConfiguration`) because only the extended variant supports dynamic partitioning. Inside it you need three correlated config blocks: `DynamicPartitioningConfiguration={"Enabled": True}`, a `ProcessingConfiguration` with a single `MetadataExtraction` processor whose two parameters are `MetadataExtractionQuery` (JQ expression) and `JsonParsingEngine` (`JQ-1.6`), and a `Prefix` that includes `!{partitionKeyFromQuery:event_type}` plus a `!{timestamp:yyyy-MM-dd}` token. Set `BufferingHints.IntervalInSeconds=60`; lower values are rejected for dynamic-partitioning streams.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2:

```python
firehose = boto3.client(
    "firehose",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

bucket_arn = f"arn:aws:s3:::{BUCKET_NAME}"
role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"

prefix = "event_type=!{partitionKeyFromQuery:event_type}/dt=!{timestamp:yyyy-MM-dd}/"
error_output_prefix = "errors/!{firehose:error-output-type}/dt=!{timestamp:yyyy-MM-dd}/"

extended_s3_config = {
    "RoleARN": role_arn,
    "BucketARN": bucket_arn,
    "Prefix": prefix,
    "ErrorOutputPrefix": error_output_prefix,
    "CompressionFormat": "Snappy",
    "BufferingHints": {"SizeInMBs": 64, "IntervalInSeconds": 60},
    "DynamicPartitioningConfiguration": {
        "Enabled": True,
        "RetryOptions": {"DurationInSeconds": 300},
    },
    "ProcessingConfiguration": {
        "Enabled": True,
        "Processors": [
            {
                "Type": "MetadataExtraction",
                "Parameters": [
                    {
                        "ParameterName": "MetadataExtractionQuery",
                        "ParameterValue": "{event_type: .event_type}",
                    },
                    {"ParameterName": "JsonParsingEngine", "ParameterValue": "JQ-1.6"},
                ],
            }
        ],
    },
    "CloudWatchLoggingOptions": {
        "Enabled": True,
        "LogGroupName": LOG_GROUP_NAME,
        "LogStreamName": "S3Delivery",
    },
}

firehose.create_delivery_stream(
    DeliveryStreamName=STREAM_NAME,
    DeliveryStreamType="DirectPut",
    ExtendedS3DestinationConfiguration=extended_s3_config,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

If `firehose.create_delivery_stream` returns `InvalidArgumentException: The role ARN ... is not valid` the IAM role propagation has not finished. Re-run after another 15 seconds. The role exists; Firehose just cannot see it yet.

</details>


**Wallet check.** Still well under a penny. Firehose has no per-stream provisioning fee; you only pay per GB ingested. The stream is sitting idle waiting for events. The ACTIVE-state polling above ran two `describe_delivery_stream` calls; AWS does not bill control-plane describes. Your morning coffee continues to dwarf this lab.


## Task 3: Run a producer that emits 500 JSON events and prove they land in S3

The pipe is built. Now exercise it. Write 500 JSON records, batched into five `put_record_batch` calls of 100 records each, then wait for Firehose to flush its 60-second buffer.

Each event payload carries an `event_type` field set to one of `page_view`, `add_to_cart`, or `purchase` (randomly selected) plus a few realistic-looking fields like `user_id`, `product_id`, and an ISO timestamp. The JSON for one event looks like:

```json
{"event_id": 42, "event_type": "add_to_cart", "user_id": "u-37", "product_id": "p-128", "ts": "2026-05-11T16:42:01.123456+00:00"}
```

Each record's `Data` field is the JSON string followed by a newline, encoded to bytes. The newline matters because Firehose treats record boundaries this way for JSON in S3 output; without the newline you lose the partition routing on the boundary record between two batches.

Record the producer wall-clock start time before the first `put_record_batch` call. Checkpoint 3 reads `PRODUCER_START_TIME` from the module namespace and computes earliest-object-`LastModified` minus producer start. The Firehose buffer hint is 60 seconds at low volume, so the validator allows up to 90 seconds for clock skew.

After the producer finishes its five batches, poll `s3.list_objects_v2` until at least one object lands or 120 seconds elapse. If 120 seconds pass with no objects, the checkpoint fails and surfaces the most likely cause (missing dynamic partitioning, wrong role trust policy, or a JQ query that does not match the JSON).


In [ ]:
# NBVAL_SKIP
# Task 3: Emit 500 JSON events in five batches of 100, then wait for the
# Firehose buffer to flush to S3.

firehose = boto3.client(
    "firehose",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)


def make_event(i: int) -> dict:
    return {
        "event_id": i,
        "event_type": random.choice(EVENT_TYPES),
        "user_id": f"u-{random.randint(1, 200)}",
        "product_id": f"p-{random.randint(1, 1000)}",
        "ts": datetime.now(timezone.utc).isoformat(),
    }


def to_record(event: dict) -> dict:
    return {"Data": (json.dumps(event) + "\n").encode("utf-8")}


# Record producer start time BEFORE the first batch. Checkpoint 3 reads this
# from the module namespace.
PRODUCER_START_TIME = datetime.now(timezone.utc)

# YOUR CODE: Emit 500 records in 5 batches of 100. For each batch:
#   1. Build records = [to_record(make_event(i)) for i in range(start, start+100)]
#   2. Call firehose.put_record_batch(DeliveryStreamName=STREAM_NAME, Records=records).
#   The producer prints progress after each batch.

print(f"Producer wrote 500 events at {PRODUCER_START_TIME.isoformat()}")
print("Asking Firehose to put on her reading glasses and shuffle the events to S3...")

# Wait up to 120 seconds for the first object to land. Firehose with dynamic
# partitioning flushes on a 60-second buffer hint at low volume; the extra 60
# seconds covers per-partition flushes and clock skew.
deadline = time.time() + 120
first_object_seen = False
while time.time() < deadline:
    resp = s3.list_objects_v2(Bucket=BUCKET_NAME, MaxKeys=1)
    if resp.get("Contents"):
        first_object_seen = True
        break
    time.sleep(5)

if not first_object_seen:
    print("WARNING: 120 seconds elapsed with no S3 object. The most likely")
    print("causes are: dynamic partitioning is disabled, the JQ MetadataExtractionQuery")
    print("does not match your event_type field, or the role trust policy is wrong.")
    print("Check Hint 1 below and re-run.")
else:
    elapsed = (datetime.now(timezone.utc) - PRODUCER_START_TIME).total_seconds()
    print(f"First object landed inside {elapsed:.1f} seconds. Buffer flushed.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: at least 50 objects land in S3 inside 90 seconds of the
# producer start time, and they carry Snappy compression markers.

def checkpoint_3(session):
    try:
        if PRODUCER_START_TIME is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "PRODUCER_START_TIME is None. Run the producer cell above "
                    "before this checkpoint."
                ),
            )

        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        # Paginate through every object that landed since the producer started.
        # The list ignores the bucket's tagging metadata; only objects matter.
        paginator = s3_client.get_paginator("list_objects_v2")
        post_objects: list[dict] = []
        for page in paginator.paginate(Bucket=BUCKET_NAME):
            for obj in page.get("Contents", []):
                last_modified = obj["LastModified"]
                if last_modified >= PRODUCER_START_TIME:
                    post_objects.append(obj)

        if len(post_objects) < 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "No objects landed in the bucket since the producer started. "
                    "Check that the role trust policy names firehose.amazonaws.com, "
                    "the inline policy grants s3:PutObject on the bucket /*, and "
                    "DynamicPartitioningConfiguration.Enabled is True."
                ),
            )

        earliest = min(post_objects, key=lambda o: o["LastModified"])
        latency = (earliest["LastModified"] - PRODUCER_START_TIME).total_seconds()
        if latency > 90:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Earliest object landed {latency:.1f}s after producer start. "
                    f"The Firehose buffer hint is 60s and the validator allows "
                    f"30s of tolerance. Re-run the producer; transient delays "
                    f"can push the first batch past 90s."
                ),
            )

        # Snappy compression marker. Firehose with CompressionFormat=Snappy
        # writes object keys ending in .snappy.
        bad_keys = [
            o["Key"] for o in post_objects
            if not o["Key"].endswith(".snappy") and not o["Key"].startswith("errors/")
        ]
        if bad_keys:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Some objects do not end in .snappy: {bad_keys[:3]}... "
                    f"Recreate the stream with CompressionFormat=Snappy."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

The validator wants two things: at least one object landed since `PRODUCER_START_TIME`, and the earliest object's `LastModified` is within 90 seconds of producer start. If the validator reports zero objects, the producer never wrote anything that Firehose accepted; check the stream status and the role. If it reports the wrong latency, the buffer is set too high or the network path is slow; the buffer hint is 60 seconds and re-running usually clears transient delays.

</details>

<details><summary>Hint 2 (direction)</summary>

The producer emits five batches of 100 records via `firehose.put_record_batch`. Each record's `Data` must be the JSON string for one event with a trailing newline, encoded to bytes. Forgetting the newline does not break Firehose, but it concatenates events into one logical line in S3 which makes downstream Athena parsing brittle. Forgetting `Data` (sending a dict instead) raises a parameter-validation error. Capture `PRODUCER_START_TIME = datetime.now(timezone.utc)` BEFORE the first batch call so the latency check measures from the right anchor.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 3:

```python
firehose = boto3.client(
    "firehose",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)


def make_event(i):
    return {
        "event_id": i,
        "event_type": random.choice(EVENT_TYPES),
        "user_id": f"u-{random.randint(1, 200)}",
        "product_id": f"p-{random.randint(1, 1000)}",
        "ts": datetime.now(timezone.utc).isoformat(),
    }


def to_record(event):
    return {"Data": (json.dumps(event) + "\n").encode("utf-8")}


PRODUCER_START_TIME = datetime.now(timezone.utc)

for batch_index in range(5):
    start = batch_index * 100
    records = [to_record(make_event(start + i)) for i in range(100)]
    resp = firehose.put_record_batch(
        DeliveryStreamName=STREAM_NAME, Records=records
    )
    failed = resp.get("FailedPutCount", 0)
    print(f"Batch {batch_index + 1}: 100 records sent, {failed} failed.")
```

If `FailedPutCount > 0` on any batch, retry just the failed records by inspecting `resp["RequestResponses"]` for entries with `ErrorCode`. At low volume this is unusual; AWS occasionally returns throttling on the first batch when a new stream just went ACTIVE.

</details>


**Wallet check.** Producer just streamed roughly 100 KB total. Firehose bills at $0.029 per GB ingested, so this batch cost about $0.0000029 (three-thousandths of a cent, rounded up). Your S3 PUT count is 5 to 20 objects depending on partition flush behavior; that fits inside the always-free S3 request budget. CloudWatch Logs for Firehose error log is empty because nothing errored. You would have to retry this lab 350,000 times to cross a penny.


## Task 4: Verify the partition layout matches event_type and UTC date

Producer ran. Objects landed. Now prove the partition routing is correct.

Every object key in the bucket must match the pattern:

```
event_type=(page_view|add_to_cart|purchase)/dt=YYYY-MM-DD/<firehose-generated-suffix>.snappy
```

Concretely, an example key looks like:

```
event_type=add_to_cart/dt=2026-05-11/labrun-kinesis-streaming-ingestion-stream-1-2026-05-11-16-43-12-a1b2c3d4.snappy
```

The validator below uses a single regex to assert three properties at once:
1. The first prefix segment is `event_type=` followed by one of the three known values.
2. The second prefix segment is `dt=` followed by `YYYY-MM-DD`.
3. At least one object exists under each of the three `event_type` partitions.

There is nothing for you to code in this task. The checkpoint runs against data the producer already emitted. If Checkpoint 3 passed but Checkpoint 4 fails, the cause is almost always the JQ MetadataExtractionQuery on the stream not matching the JSON shape; events were buffered into the literal `!{partitionKeyFromQuery:event_type}` prefix and never resolved to the three real partitions.


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: every object key matches event_type=<value>/dt=YYYY-MM-DD/...
# and at least one object exists under each of the three event_type partitions.

KEY_PATTERN = re.compile(
    r"^event_type=(page_view|add_to_cart|purchase)/dt=\d{4}-\d{2}-\d{2}/[^/]+$"
)


def checkpoint_4(session):
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        paginator = s3_client.get_paginator("list_objects_v2")
        seen_event_types: set[str] = set()
        mismatched: list[str] = []
        total = 0
        for page in paginator.paginate(Bucket=BUCKET_NAME):
            for obj in page.get("Contents", []):
                key = obj["Key"]
                if key.startswith("errors/"):
                    continue
                total += 1
                m = KEY_PATTERN.match(key)
                if not m:
                    mismatched.append(key)
                    continue
                seen_event_types.add(m.group(1))

        if total == 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "No data objects found under the bucket. Re-run the "
                    "producer cell from Task 3 first."
                ),
            )

        if mismatched:
            sample = mismatched[:3]
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{len(mismatched)} of {total} object keys do not match "
                    f"event_type=<value>/dt=YYYY-MM-DD/.... Sample: {sample}. "
                    f"The most likely cause is dynamic partitioning was not "
                    f"enabled OR the MetadataExtractionQuery JQ expression does "
                    f"not resolve event_type from the JSON record."
                ),
            )

        missing = set(EVENT_TYPES) - seen_event_types
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No objects landed under these event_type partitions: "
                    f"{sorted(missing)}. The producer is supposed to randomize "
                    f"across all three values. Re-run the producer and wait "
                    f"for another buffer flush."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

The validator runs three checks: every key matches the regex, no key lands outside the partition structure, and all three event_type partitions have at least one object. If you see mismatched keys, the partition tokens never resolved; check the delivery stream's `ProcessingConfiguration`. If you see missing event_type partitions, run the producer again; Firehose may have flushed a partial sample and the next buffer carries the rest.

</details>

<details><summary>Hint 2 (direction)</summary>

If keys come out as `!{partitionKeyFromQuery:event_type}/dt=...` the JQ extractor never ran. That means the stream was created without the MetadataExtraction processor or with a malformed `MetadataExtractionQuery`. The right JQ expression for events shaped `{"event_type": "page_view", ...}` is `{event_type: .event_type}`. The JsonParsingEngine parameter must be `JQ-1.6`. To fix the stream without recreating it from scratch, run `firehose.update_destination` with the corrected `ExtendedS3DestinationUpdate.ProcessingConfiguration`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

There is no code to write for this checkpoint; the producer already emitted the records in Task 3. If you need to fix a misconfigured stream, here is the corrective update call:

```python
firehose = boto3.client(
    "firehose",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

desc = firehose.describe_delivery_stream(DeliveryStreamName=STREAM_NAME)
version_id = desc["DeliveryStreamDescription"]["VersionId"]
destination_id = desc["DeliveryStreamDescription"]["Destinations"][0]["DestinationId"]

firehose.update_destination(
    DeliveryStreamName=STREAM_NAME,
    CurrentDeliveryStreamVersionId=version_id,
    DestinationId=destination_id,
    ExtendedS3DestinationUpdate={
        "ProcessingConfiguration": {
            "Enabled": True,
            "Processors": [
                {
                    "Type": "MetadataExtraction",
                    "Parameters": [
                        {
                            "ParameterName": "MetadataExtractionQuery",
                            "ParameterValue": "{event_type: .event_type}",
                        },
                        {
                            "ParameterName": "JsonParsingEngine",
                            "ParameterValue": "JQ-1.6",
                        },
                    ],
                }
            ],
        },
    },
)
```

Then re-run the producer cell. Objects emitted before the update keep their broken prefix and remain in S3; only post-update objects route correctly. The cleanup cell deletes the bucket so the broken keys do not survive the session.

</details>


**Wallet check.** A handful of S3 `list_objects_v2` calls and one regex scan over the producer output. All control-plane. Free-tier S3 absorbs the listing volume. The notebook's running total is still under a penny. The cleanup cell next will shut everything down.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order. Lab 03 has no critical (hourly-billed) resources.
#
# labrun-checks 0.3.0 ships AWS adapters for s3_bucket and iam_role.
# It does NOT yet support firehose_delivery_stream or cloudwatch_log_group.
# A LabAdapter subclass wraps AwsCleanupAdapter and adds the two missing
# resource types as an inline fallback. When the package promotes these
# adapters in a future release, the LabAdapter wrapper can be removed and
# run_cleanup called against the default adapter.

import sys
from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter:
    '''AwsCleanupAdapter wrapper that adds firehose_delivery_stream
    and cloudwatch_log_group support.'''

    def __init__(self) -> None:
        self._aws = AwsCleanupAdapter()

    def delete_resource(self, credentials: dict, resource) -> None:
        if resource.type == "firehose_delivery_stream":
            client = boto3.client(
                "firehose",
                aws_access_key_id=credentials["aws_access_key_id"],
                aws_secret_access_key=credentials["aws_secret_access_key"],
                aws_session_token=credentials.get("aws_session_token"),
                region_name=credentials.get("region", REGION),
            )
            try:
                client.delete_delivery_stream(
                    DeliveryStreamName=resource.id, AllowForceDelete=True
                )
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return
                raise
            return
        if resource.type == "cloudwatch_log_group":
            client = boto3.client(
                "logs",
                aws_access_key_id=credentials["aws_access_key_id"],
                aws_secret_access_key=credentials["aws_secret_access_key"],
                aws_session_token=credentials.get("aws_session_token"),
                region_name=credentials.get("region", REGION),
            )
            try:
                client.delete_log_group(logGroupName=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return
                raise
            return
        return self._aws.delete_resource(credentials, resource)

    def describe_resource(self, credentials: dict, resource) -> bool:
        if resource.type == "firehose_delivery_stream":
            client = boto3.client(
                "firehose",
                aws_access_key_id=credentials["aws_access_key_id"],
                aws_secret_access_key=credentials["aws_secret_access_key"],
                aws_session_token=credentials.get("aws_session_token"),
                region_name=credentials.get("region", REGION),
            )
            try:
                desc = client.describe_delivery_stream(DeliveryStreamName=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return False
                raise
            status = desc["DeliveryStreamDescription"]["DeliveryStreamStatus"]
            # DELETING means the async delete is in flight; treat as gone so
            # Phase 2 verification does not flag it.
            return status != "DELETING"
        if resource.type == "cloudwatch_log_group":
            client = boto3.client(
                "logs",
                aws_access_key_id=credentials["aws_access_key_id"],
                aws_secret_access_key=credentials["aws_secret_access_key"],
                aws_session_token=credentials.get("aws_session_token"),
                region_name=credentials.get("region", REGION),
            )
            try:
                resp = client.describe_log_groups(
                    logGroupNamePrefix=resource.id, limit=5
                )
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return False
                raise
            names = [g["logGroupName"] for g in resp.get("logGroups", [])]
            return resource.id in names
        return self._aws.describe_resource(credentials, resource)

    def tag_scan(self, credentials: dict, lab_slug: str, region: str) -> list[str]:
        return self._aws.tag_scan(credentials, lab_slug, region)


# Empty the S3 bucket before run_cleanup tears it down. S3 buckets cannot be
# deleted while they contain objects; the bundled s3_bucket adapter deletes
# one page of objects only, and Firehose may have written hundreds of small
# Snappy files spread across event_type partitions. Empty here so the adapter
# call succeeds on the first try.
print("Emptying the S3 bucket before teardown...")
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NAME):
        keys = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if keys:
            s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": keys})
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchBucket":
        print(f"Bucket emptying ran into: {e}. Continuing to cleanup.")

result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: about $0.01.** Firehose ingestion of ~100 KB across the producer's 500 events bills under a penny. S3 storage and request volume fits inside the always-free tier. CloudWatch Logs only carried error logs (zero on a clean run). The stream, log group, role, and bucket are torn down by the cleanup cell above so nothing is still accruing. Your morning coffee outpriced this lab by two orders of magnitude.


## These are not graded. They are for you.

Three questions worth sitting with before the exam.

1. The dynamic partitioning pipeline you just built routes by `event_type` AND UTC date. If you re-ran this lab a year from now with the same producer, the data would land in `event_type=<value>/dt=2027-05-11/` and Athena's partition projection would naturally degrade in performance if the year/month split were absent. Walk through how you would change the Firehose `Prefix` and the downstream Athena table to keep query speed flat as the partition cardinality grows over time. This maps to DEA-C01 Domain 1 expectations on partition strategy under high cardinality.

2. The CTO asks why this lab used Kinesis Data Firehose instead of Kinesis Data Streams + a Lambda consumer + a manual S3 writer. Identify three scenarios where the latter is the correct choice and three where Firehose is. The two services solve overlapping problems and the exam tests whether you can recognize when each is better. Lab 08 covers the Lambda consumer pattern end to end.

3. Firehose buffer hints are 60 seconds OR 5 MB at low volume on dynamic-partitioning streams, whichever fills first. At what producer volume does the buffer flip from time-based to size-based? Estimate the volume in events-per-second assuming each event averages 200 bytes. This is a back-of-the-envelope question DEA-C01 likes to ask about latency-vs-batch-size trade-offs.

## Exam-Style Practice

**Q1.** A data engineer configures a Kinesis Data Firehose delivery stream with dynamic partitioning enabled and `Prefix="event_type=!{partitionKeyFromQuery:event_type}/dt=!{timestamp:yyyy-MM-dd}/"`. The stream is `ACTIVE` and the producer sends 500 JSON events tagged with `event_type` values. All 500 events land under the literal key prefix `event_type=!{partitionKeyFromQuery:event_type}/` with the token unresolved. What is the most likely cause?

A) The IAM role attached to the delivery stream is missing `s3:PutObject` on the bucket.

B) The `ProcessingConfiguration` is missing or does not include a `MetadataExtraction` processor with `JsonParsingEngine=JQ-1.6`.

C) The Firehose buffer hint is set above 60 seconds, so the partition tokens did not resolve in time.

D) The `DeliveryStreamType` is `KinesisStreamAsSource` instead of `DirectPut`.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because a missing `s3:PutObject` permission would surface as Firehose error logs and zero objects in the bucket, not as objects with unresolved partition tokens. Objects landing means writes are succeeding.
- B) Right because the `partitionKeyFromQuery:event_type` token only resolves when a `MetadataExtraction` processor with the JQ engine extracts `event_type` from each record. Without the processor, the token stays as a literal string in the key.
- C) Wrong because buffer hints control flush timing, not token resolution. Token resolution happens at processing time, before the buffer flushes to S3.
- D) Wrong because the `DeliveryStreamType` controls the source, not the destination prefix behavior. `DirectPut` is correct for a producer that calls `put_record_batch` and is unrelated to the partitioning bug.

</details>

**Q2.** A team needs to ingest 50,000 JSON events per second from a fleet of mobile clients into S3, partitioned by event type. The team plans to use Kinesis Data Firehose with dynamic partitioning enabled. Which Firehose configuration is the LEAST appropriate for this scale?

A) `DeliveryStreamType="DirectPut"` with the mobile clients calling `put_record_batch` in batches of 500 records.

B) `BufferingHints={"SizeInMBs": 128, "IntervalInSeconds": 60}` to amortize the per-object S3 write cost over larger batches.

C) `CompressionFormat="Snappy"` to reduce S3 storage cost without giving up readable downstream parsing.

D) `BufferingHints={"SizeInMBs": 1, "IntervalInSeconds": 15}` to keep per-event latency under 30 seconds at all times.

<details><summary>Show answer</summary>

**D**.

- A) Right configuration. `DirectPut` is the correct source for clients that write records directly; batching at the client reduces API call overhead.
- B) Right configuration. At 50K events per second with ~200-byte payloads, the 5 MB partition buffer fills in well under a second, and 128 MB is the maximum allowed on Firehose; this is the right knob at this scale.
- C) Right configuration. Snappy compression reduces S3 storage and downstream Athena scan cost without requiring a decompression step in the producer.
- D) Wrong. Firehose with dynamic partitioning rejects `IntervalInSeconds` below 60. Even if it accepted lower values, a 1 MB buffer at this volume would produce thousands of small files per minute and trigger S3 PUT request throttling. The right answer here is to accept the 60-second latency floor.

</details>

**Q3.** A data engineer creates a Firehose delivery stream and immediately calls `firehose.put_record_batch` from a notebook. The call returns `InvalidArgumentException: The role ARN ... is not valid`. The role exists in IAM and the trust policy is correct. What is the right next step?

A) Recreate the role with a different name; AWS sometimes caches deleted ARNs.

B) Wait 15 to 30 seconds for IAM propagation and retry the call; the role exists but Firehose has not seen it yet.

C) Attach `iam:PassRole` to the IAM user running the notebook; without it Firehose cannot read the role.

D) Switch the region to `us-east-1` because Firehose only validates roles in that region.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. AWS does not cache deleted ARNs on the Firehose side, and renaming the role does not address the propagation lag.
- B) Right. IAM role creation propagates to the broader AWS plane asynchronously. Firehose validates the role at create-stream and put-record time; if either runs before propagation finishes, the role appears not to exist. The standard recovery is to wait 15 to 30 seconds and retry. The lab's setup cell builds this wait into the role-creation step for exactly this reason.
- C) Wrong. `iam:PassRole` is required on the user that creates the delivery stream, but the error in this scenario is on `put_record_batch`, not on the create call. The `PassRole` permission gates role attachment, not record ingestion.
- D) Wrong. Firehose is regional and validates roles in whatever region the stream lives in. There is no special role-validation behavior in `us-east-1`.

</details>
